at top right Genie- 
prepare a standard calendar table as "project1_gs_caresync.silver.calendar" table. it shall have columns like datekey, date, year, month, week, quarter etc and all the main standard calendar columns. It shall have data from "1940-01-01 to 2030-12-31" date.

# Calendar Dimension Table

Builds the standard calendar table `project1_gs_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
%sql
CREATE OR REPLACE TABLE project1_gs_caresync.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                         AS datekey,
  d                                                               AS date,
  -- Year
  year(d)                                                         AS year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0
       THEN true ELSE false END                                   AS is_leap_year,
  CAST(trunc(d, 'YEAR') AS DATE)                                  AS first_day_of_year,
  make_date(year(d), 12, 31)                                      AS last_day_of_year,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END                       AS half_year,
  concat('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END)          AS half_year_name,
  -- Quarter
  quarter(d)                                                      AS quarter,
  concat('Q', quarter(d))                                         AS quarter_name,
  concat(year(d), '-Q', quarter(d))                               AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)                          AS first_day_of_quarter,
  date_add(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), -1) AS last_day_of_quarter,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1         AS day_of_quarter,
  -- Month
  month(d)                                                        AS month,
  date_format(d, 'MMMM')                                          AS month_name,
  date_format(d, 'MMM')                                           AS month_short_name,
  date_format(d, 'yyyy-MM')                                       AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                           AS year_month_key,
  CAST(trunc(d, 'MM') AS DATE)                                    AS first_day_of_month,
  last_day(d)                                                     AS last_day_of_month,
  day(last_day(d))                                                AS days_in_month,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(d)                                                   AS week_of_year,
  extract(YEAROFWEEK FROM d)                                      AS iso_year,
  concat(extract(YEAROFWEEK FROM d), '-W', lpad(weekofyear(d), 2, '0')) AS year_week,
  CAST(ceil(day(d) / 7.0) AS INT)                                 AS week_of_month,
  CAST(date_trunc('WEEK', d) AS DATE)                             AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)                AS week_end_date,
  -- Day
  day(d)                                                          AS day_of_month,
  dayofyear(d)                                                    AS day_of_year,
  extract(DAYOFWEEK_ISO FROM d)                                   AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                                          AS day_name,
  date_format(d, 'EEE')                                           AS day_short_name,
  -- Flags
  extract(DAYOFWEEK_ISO FROM d) IN (6, 7)                         AS is_weekend,
  extract(DAYOFWEEK_ISO FROM d) NOT IN (6, 7)                     AS is_weekday,
  d = CAST(trunc(d, 'MM') AS DATE)                                AS is_month_start,
  d = last_day(d)                                                 AS is_month_end,
  d = CAST(date_trunc('QUARTER', d) AS DATE)                      AS is_quarter_start,
  d = date_add(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), -1) AS is_quarter_end,
  d = CAST(trunc(d, 'YEAR') AS DATE)                              AS is_year_start,
  d = make_date(year(d), 12, 31)                                  AS is_year_end
FROM dates
ORDER BY d

In [0]:
%sql
SELECT
  count(*)                  AS total_rows,
  count(DISTINCT datekey)   AS distinct_datekeys,
  min(date)                 AS min_date,
  max(date)                 AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows
FROM project1_gs_caresync.silver.calendar

In [0]:
%sql
SELECT * FROM project1_gs_caresync.silver.calendar
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-06'
ORDER BY date

this is for Gold tables

/Workspace/caresync_repo/caresync-adb-repo/Gold/sttm_silver_to_gold.xlsx. this is an sttm documents. you shall first understand all the sheets in it. then you shall analyse schema of all source tables that are mentions, then you shall create 3 notebooks in the same folder using the details mention in the sttm document.